# Agent

Agent là một model thực hiện gọi các tool theo một vòng lặp cho đến khi hoàn thành một tác vụ được giao.

<p align="center">
    <img src="https://mintcdn.com/langchain-5e9cc07a/jtty0O--UJOKG0nK/oss/images/core_agent_loop.svg?fit=max&auto=format&n=jtty0O--UJOKG0nK&q=85&s=4b4cbb497b6273758a565de1bc90ece0" height="300">
</p>

Harness (bộ khung) bao gồm mọi thứ xung quanh vòng lặp đó: prompt, các tool và bất kỳ middleware nào giúp định hình hành vi của model.

<div class="alert alert-info">
    <b>Agent = Model + Harness</b>
    <p>Nhiệm vụ của harness: cung cấp cho model đúng context vào đúng thời điểm cho tác vụ được giao.</p>
</div>

[`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_agent?_gl=1*1lp5vq8*_gcl_au*MjA5MzEyMzM1NS4xNzg3MTg5OTgx*_ga*MTg4Njg5NDgwMS4xNzcwMzYxMzE1*_ga_47WX3HKKY2*czE3ODczODIzNzckbzQ3JGcxJHQxNzg3Mzg0MjgwJGo2MCRsMCRoMA..) là một harness có khả năng cấu hình cao. Ở mức đơn giản nhất, bạn có thể tạo một agent bằng:

In [ ]:
from langchain.agents import create_agent

agent = create_agent(model="google_genai:gemini-3.5-flash-lite")

Dựa trên nền tảng đó, bạn có thể cấu hình các thành phần cơ bản trực tiếp bằng các tham số `model=`, `tools=` và `system_prompt=`. Đối với các tính năng nâng cao hơn, hãy mở rộng harness bằng [middleware](https://docs.langchain.com/oss/python/langchain/agents#configure-the-harness).

<div class="alert alert-success">

[Deep Agents](https://docs.langchain.com/oss/python/deepagents/overview) được xây dựng dựa trên `create_agent` và đi kèm với các tính năng hữu ích thường dùng đã được lắp ráp sẵn, chẳng hạn như planning, các tool hệ thống file, subagent và memory. Hãy sử dụng `create_agent` khi bạn cần tự mình cấu hình harness.

</div>

## Các thành phần cốt lõi

<p align="center">
    <img src="https://mintcdn.com/langchain-5e9cc07a/jtty0O--UJOKG0nK/oss/images/agent_model_harness.svg?fit=max&auto=format&n=jtty0O--UJOKG0nK&q=85&s=5ac6a7e0343af7cb5ba3ca632e2224af" height="300">
</p>

### Model

Truyền một chuỗi định danh model (`"provider:model"`) hoặc một instance của model đã được khởi tạo để chọn model cho agent của bạn. Xem mục [Models](https://docs.langchain.com/oss/python/langchain/models) để biết thêm về các tham số, cách thiết lập provider và cách chọn model động.


In [2]:
from langchain.agents import create_agent

agent = create_agent(model="google_genai:gemini-3.5-flash-lite")

### Tool

Để cung cấp các tool cho agent, hãy truyền vào bất kỳ Python callable nào, LangChain tool hoặc dictionary chứa tool. Xem mục [Tools](https://docs.langchain.com/oss/python/langchain/tools) để biết cách định nghĩa tool, truy cập context và chọn tool động.

In [3]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def search(query: str) -> str:
    """Tìm kiếm thông tin."""
    return f"Kết quả cho: {query}"


agent = create_agent(model="google_genai:gemini-3.5-flash-lite", tools=[search])

### System prompt

Định hình cách agent tiếp cận các tác vụ. Tham số system prompt chấp nhận một string hoặc `SystemMessage`. Đối với các prompt động khi runtime, hãy sử dụng [middleware](https://docs.langchain.com/oss/python/langchain/middleware).

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    system_prompt="Bạn là một trợ lý hữu ích. Hãy trả lời ngắn gọn và chính xác.",
)

### Output có cấu trúc

Trả về một schema đã được validate từ agent bằng cách sử dụng `response_format=`. Xem mục [Structured output](https://docs.langchain.com/oss/python/langchain/structured-output) để biết các chiến lược và ví dụ.

In [6]:
from pydantic import BaseModel
from langchain.agents import create_agent


class Answer(BaseModel):
    summary: str
    confidence: float


agent = create_agent(model="google_genai:gemini-3.5-flash-lite", response_format=Answer)
result = agent.invoke({"messages": [{"role": "user", "content": "Tóm tắt các xu hướng AI"}]})
result["structured_response"]

Answer(summary='Các xu hướng AI hiện nay tập trung vào phát triển trí tuệ nhân tạo tạo sinh (Generative AI), mô hình ngôn ngữ lớn (LLM) đa phương thức, tự động hóa quy trình, tăng cường bảo mật và đạo đức AI, cùng với việc ứng dụng AI sâu rộng vào y tế, giáo dục và kinh doanh.', confidence=0.95)

### State của agent

Mỗi agent quản lý context thực thi của nó thông qua [`AgentState`](https://reference.langchain.com/python/langchain/agents/middleware/types/AgentState), một dictionary có định kiểu lưu trữ lịch sử hội thoại hiện tại và bất kỳ trường tùy chỉnh nào mà các tool và middleware của bạn cần.

Trường được tích hợp sẵn là:

| Trường      | Kiểu Dữ liệu        | Mô tả                                                                                                        |
| ---------- | ------------------- | ---------------------------------------------------------------------------------------------------------- |
| `messages` | `list[BaseMessage]` | Lịch sử hội thoại đầy đủ cho luồng hiện tại. Chỉ ghi thêm: các tin nhắn mới được thêm vào, không bao giờ bị ghi đè. |

`AgentState` cũng là type signature cho mọi middleware hook mang phong cách node (`before_model`, `after_model`, và tương tự). Các hook nhận state hiện tại và có thể trả về một dictionary chứa các bản cập nhật để gộp ngược lại vào state.

Để thêm các trường tùy chỉnh (ví dụ: `user_id` hoặc bộ đếm), hãy tạo subclass kế thừa từ `AgentState` và truyền subclass đó vào `create_agent` thông qua `state_schema=`:

In [7]:
from langchain.agents import AgentState, create_agent


class MyState(AgentState):
    user_id: str
    call_count: int


agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    state_schema=MyState,
)

Để biết thông tin chi tiết đầy đủ, các ví dụ, và state schema ở cấp độ middleware, hãy xem [Bộ nhớ ngắn hạn](https://docs.langchain.com/oss/python/langchain/short-term-memory#customizing-agent-memory) và [Middleware tùy chỉnh](https://docs.langchain.com/oss/python/langchain/middleware/custom#state-updates).

## Gọi thực thi

<div class="alert alert-success">

Theo dõi từng bước của vòng lặp này, gỡ lỗi các lệnh gọi tool, và đánh giá các output của agent bằng [LangSmith](https://smith.langchain.com?utm_source=docs\&utm_medium=cta\&utm_campaign=langsmith-signup\&utm_content=oss-langchain-agents). Làm theo [tracing quickstart](https://docs.langchain.com/langsmith/trace-with-langchain) để thiết lập. Chúng tôi khuyên bạn cũng nên cài đặt [LangSmith Engine](https://docs.langchain.com/langsmith/engine) - công cụ giúp theo dõi trace của bạn, phát hiện các sự cố và đề xuất cách khắc phục.

</div>

Bạn có thể invoke một agent bằng một tin nhắn. Ở phía sau, điều này sẽ truyền một bản cập nhật vào [`State`](https://docs.langchain.com/oss/python/langgraph/graph-api#state) của agent. Tất cả các agent đều bao gồm một [chuỗi các tin nhắn](https://docs.langchain.com/oss/python/langgraph/use-graph-api#messagesstate) trong state của chúng; để invoke agent, hãy truyền vào một tin nhắn mới cùng với `thread_id` để agent có thể lưu trữ và tiếp tục lịch sử hội thoại:

In [8]:
from langchain.agents import create_agent
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": str(uuid7())}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Thời tiết ở San Francisco như thế nào?"}]},
    config=config,
)

# Một lượt tiếp theo trong cùng một cuộc hội thoại: sử dụng lại cùng một thread_id để giữ lịch sử
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Ngày mai thì sao?"}]},
    config=config,
)

<div class="alert alert-info">

Việc lưu trữ lịch sử hội thoại bằng `thread_id` yêu cầu agent phải được cấu hình với một [checkpointer](https://docs.langchain.com/oss/python/langchain/long-term-memory). Khi triển khai trên [LangSmith](https://docs.langchain.com/langsmith/deployment), một checkpointer sẽ được cung cấp tự động. Nếu chạy local, bạn cần truyền nó vào một cách rõ ràng, ví dụ: `create_agent(..., checkpointer=InMemorySaver())`.

</div>

Nếu bạn cũng cần truyền các cấu hình cho mỗi lần chạy (chẳng hạn như user ID, API key hoặc feature flag) vào các tool và middleware, hãy truyền nó dưới dạng `context` cùng với `config`. Định nghĩa cấu trúc của dữ liệu đó bằng `context_schema` và truy cập nó thông qua `runtime.context`:

In [9]:
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver


@dataclass
class Context:
    user_id: str


agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[],
    context_schema=Context,
    checkpointer=InMemorySaver(),
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Thời tiết ở San Francisco như thế nào?"}]},
    config={"configurable": {"thread_id": str(uuid7())}},
    context=Context(user_id="user-123"),
)

`thread_id` giới hạn phạm vi của *cuộc hội thoại* (lịch sử tin nhắn, checkpoint), trong khi `context` mang dữ liệu *của mỗi lần chạy* để các tool và middleware của bạn đọc tại thời điểm gọi. Cả hai thường được truyền cùng nhau. Xem [tool context](https://docs.langchain.com/oss/python/langchain/tools#context) và [Runtime](https://docs.langchain.com/oss/python/langchain/runtime) để biết thêm.

## Streaming

`invoke` trả về response cuối cùng khi kết thúc một lần chạy. Nếu agent thực thi nhiều lệnh gọi tool, người dùng thường cần được cập nhật tiến độ trước khi hoàn tất. Hãy sử dụng streaming để hiển thị các tin nhắn trung gian và hoạt động của tool ngay khi chúng diễn ra.

In [12]:
from langchain.messages import AIMessage, HumanMessage
from langchain.agents import create_agent


agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
)

stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "Tìm kiếm tin tức về AI và tóm tắt những gì tìm được"}]},
    version="v3",
)
for snapshot in stream.values:
    # Mỗi snapshot chứa toàn bộ state tại thời điểm đó
    latest_message = snapshot["messages"][-1]
    if latest_message.content:
        if isinstance(latest_message, HumanMessage):
            print(f"Người dùng: {latest_message.content}")
        elif isinstance(latest_message, AIMessage):
            print(f"Agent: {latest_message.content}")
    elif latest_message.tool_calls:
        print(f"Đang gọi các tool: {[tc['name'] for tc in latest_message.tool_calls]}")

Người dùng: Tìm kiếm tin tức về AI và tóm tắt những gì tìm được
Agent: [{'type': 'text', 'text': 'Dưới đây là tổng hợp các tin tức nổi bật và quan trọng nhất về Trí tuệ Nhân tạo (AI) trong thời gian gần đây (cập nhật đến tháng **5/2024**):\n\n### 1. Cuộc đua Mô hình Ngôn ngữ Lớn (LLM): OpenAI vs. Google vs. Meta\n* **OpenAI ra mắt GPT-4o ("Omni"):** Đây là bước tiến lớn khi GPT-4o có khả năng xử lý nguyên bản (natively multimodal) giọng nói, văn bản và hình ảnh thời gian thực. Phản hồi bằng giọng nói của AI này có độ trễ cực thấp (chỉ tương đương con người - khoảng 232 mili-giây), mang lại cảm giác trò chuyện tự nhiên như đang gọi điện thoại.\n* **Google I/O 2024:** Google công bố tích hợp sâu AI **Gemini** vào toàn bộ hệ sinh thái của họ (Android, Search, Workspace). Đáng chú ý là tính năng **Project Astra** (trợ lý ảo AI đa phương thức nhìn và hiểu thế giới qua camera điện thoại/kính thông minh) và khả năng tạo video ngắn từ văn bản thông qua **Veo**.\n* **Meta AI:** Tiếp tục đẩy mạn

<div class="alert alert-success">

Để biết các chế độ streaming, các loại event và các pattern UI, xem [Streaming](https://docs.langchain.com/oss/python/langchain/streaming).

</div>

## Cấu hình harness

`create_agent` có khả năng mở rộng cao. Middleware là thành phần cơ bản để tùy chỉnh: mỗi phần xử lý một chức năng nhất định, gắn vào vòng lặp agent đúng thời điểm, và dễ dàng kết hợp với các middleware khác. Bạn chỉ cần lấy chính xác những gì use case của mình cần và bỏ qua phần còn lại.

Các pattern phổ biến được xây dựng sẵn dưới dạng first-class middleware. Bạn có thể xây dựng bất kỳ thứ gì khác dưới dạng [middleware tùy chỉnh](https://docs.langchain.com/oss/python/langchain/middleware/custom).

<img src="https://mintcdn.com/langchain-5e9cc07a/jtty0O--UJOKG0nK/oss/images/agent_harness_capabilities.svg?fit=max&auto=format&n=jtty0O--UJOKG0nK&q=85&s=0ff671d72badd0844826660dfcb04391" height="300">

Khi agent đảm nhận những công việc phức tạp, chúng cần được hỗ trợ trong một vài lĩnh vực chính. Hệ sinh thái middleware cung cấp:

- **Môi trường thực thi**: Các tool, hệ thống file, sandbox và thực thi code
- **Quản lý context**: Tóm tắt, memory, skill và caching prompt
- **Lập kế hoạch và ủy quyền**: Danh sách công việc và subagent để xử lý các công việc song song, độc lập
- **Khả năng chịu lỗi**: Thử lại, dự phòng và giới hạn số lần gọi
- **Rào chắn bảo vệ**: Phát hiện PII và kiểm soát nội dung
- **Điều hướng**: Sự phê duyệt của con người trước các hành động có tác động lớn

<div class="alert alert-success">

`create_deep_agent` lắp ráp sẵn stack này cho các tác vụ nghiên cứu và lập trình chạy thời gian dài (filesystem, summarization, subagent và prompt caching được bao gồm theo mặc định). Xem [Deep Agents](https://docs.langchain.com/oss/python/deepagents/harness) để biết toàn bộ harness được tạo sẵn.

</div>

### Môi trường thực thi

Các agent đặc biệt hữu ích khi chúng có thể thực hiện hành động thay vì chỉ tạo ra văn bản. Môi trường thực thi cung cấp cho agent một workspace: các tool mà nó có thể gọi, một hệ thống file để đọc và ghi file qua các lượt tương tác, và khả năng thực thi code để chạy các script hoặc lệnh shell.

In [13]:
from langchain.agents import create_agent
from deepagents.backends import StateBackend
from deepagents.middleware import FilesystemMiddleware


agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    middleware=[FilesystemMiddleware(backend=StateBackend())],
)

Xem [`FilesystemMiddleware`](https://reference.langchain.com/python/deepagents/middleware/filesystem/FilesystemMiddleware), [Sandboxes](https://docs.langchain.com/oss/python/deepagents/sandboxes), [Interpreters](https://docs.langchain.com/oss/python/deepagents/interpreters).

### Quản lý ngữ cảnh

Mỗi lần gọi model đều có một context window cố định. Khi agent chạy, window đó sẽ được lấp đầy bởi lịch sử tích lũy, kết quả từ tool và các bước trung gian. Tính năng tóm tắt nén lịch sử trước khi đầy tràn; memory load các chỉ thị cố định lúc khởi động để bảo toàn kiến thức qua các session khác nhau; các skill phơi bày kiến thức chuyên ngành theo yêu cầu thay vì phải load mọi thứ ngay từ đầu.

In [ ]:
from langchain.agents import create_agent
from deepagents.backends import StateBackend
from deepagents.middleware import FilesystemMiddleware, MemoryMiddleware, SkillsMiddleware, SummarizationMiddleware

backend = StateBackend()
model = "google_genai:gemini-3.5-flash-lite"

agent = create_agent(
    model=model,
    middleware=[
        FilesystemMiddleware(backend=backend),
        SummarizationMiddleware(model=model, backend=backend),
        MemoryMiddleware(backend=backend, sources=["./AGENTS.md"]),
        SkillsMiddleware(backend=backend, sources=["./skills/"]),
    ],
)

Xem [`SummarizationMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/summarization/SummarizationMiddleware), [`MemoryMiddleware`](https://reference.langchain.com/python/deepagents/middleware/memory/MemoryMiddleware), [Skills](https://docs.langchain.com/oss/python/langchain/multi-agent/skills), [Context engineering](https://docs.langchain.com/oss/python/deepagents/context-engineering).

### Lập kế hoạch và ủy quyền

Các tác vụ phức tạp thường vượt quá khả năng xử lý của một context window. Việc ủy quyền cho phép agent chính chia nhỏ công việc, giao chúng cho các subagent để mỗi subagent chạy trong context độc lập của riêng nó, và giúp agent chính tập trung vào việc điều phối thay vì thực thi. Các công việc có thể chạy song song; context của agent chính được giữ sạch sẽ.

In [15]:
from deepagents.backends import StateBackend
from deepagents.middleware import FilesystemMiddleware
from deepagents.middleware.subagents import SubAgentMiddleware
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.tools import tool


@tool
def search(query: str) -> str:
    """Tìm kiếm một truy vấn và trả về bản tóm tắt ngắn gọn."""
    return f"Kết quả tìm kiếm cho: {query}"


backend = StateBackend()

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[search],
    middleware=[
        FilesystemMiddleware(backend=backend),
        TodoListMiddleware(),
        SubAgentMiddleware(
            backend=backend,
            subagents=[
                {
                    "name": "researcher",
                    "description": "Tìm kiếm và trả về một bản tóm tắt có cấu trúc.",
                    "system_prompt": "Sử dụng công cụ tìm kiếm để nghiên cứu câu hỏi và tóm tắt các ý chính.",
                    "tools": [search],
                    "model": "anthropic:claude-sonnet-4-6",
                    "middleware": [],
                }
            ],
        ),
    ],
)

Xem [Subagents](https://docs.langchain.com/oss/python/langchain/multi-agent/subagents).

### Đặt tên cho agent của bạn

Bạn có thể tùy chọn sử dụng một định danh cho agent. Điều này đặc biệt hữu ích khi nhúng agent dưới dạng một subgraph trong các hệ thống [multi-agent](https://docs.langchain.com/oss/python/langchain/multi-agent).

In [ ]:
agent = create_agent(model="google_genai:gemini-3.5-flash-lite", name="research_assistant")

### Khả năng chịu lỗi

Các agent trên môi trường production thường gặp phải những lỗi hiếm khi xuất hiện trong môi trường phát triển: rate limit, model timeout, lỗi API tạm thời. Các middleware chịu lỗi sẽ xử lý những vấn đề này ở cấp độ cơ sở hạ tầng, do đó các tool và logic nghiệp vụ của bạn không cần dùng try/catch cho mỗi lần gọi.

In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware, ToolRetryMiddleware
from langchain.tools import tool


@tool
def search(query: str) -> str:
    """Tìm kiếm một truy vấn và trả về bản tóm tắt ngắn gọn."""
    return f"Kết quả tìm kiếm cho: {query}"


agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[search],
    middleware=[
        ModelRetryMiddleware(max_retries=3),
        ToolRetryMiddleware(max_retries=2),
    ],
)

Xem [`ModelRetryMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/model_retry/ModelRetryMiddleware), [`ToolRetryMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/tool_retry/ToolRetryMiddleware), [Prebuilt middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in).

### Rào chắn bảo vệ

Một số chính sách không thể chỉ nằm trong prompt - chúng cần được thực thi một cách xác định bất kể model thực hiện hành động gì. Guardrail can thiệp vào dữ liệu khi nó chảy qua vòng lặp của agent, áp dụng các quy tắc tuân thủ hoặc chính sách nội dung trước khi kết quả của tool đi vào context của model.

In [18]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.tools import tool


@tool
def search(query: str) -> str:
    """Tìm kiếm một truy vấn và trả về bản tóm tắt ngắn gọn."""
    return f"Kết quả tìm kiếm cho: {query}"


agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[search],
    middleware=[PIIMiddleware("email")],
)

Xem [`PIIMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/pii/PIIMiddleware), [Prebuilt middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in).

### Điều hướng

Tính tự chủ hoàn toàn không phải lúc nào cũng phù hợp. Việc điều hướng cho phép bạn đặt sự can thiệp của con người tại các điểm quyết định cụ thể - trước các lệnh ghi mang tính phá hủy, các lệnh gọi API tốn kém hoặc bất cứ điều gì cần sự phán đoán - mà không phải cấu trúc lại agent của bạn. Agent sẽ tạm dừng và chờ đợi; con người thực hiện phê duyệt, chỉnh sửa hoặc từ chối; sau đó quá trình thực thi mới tiếp tục.

In [19]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool


@tool
def search(query: str) -> str:
    """Tìm kiếm một truy vấn và trả về bản tóm tắt ngắn gọn."""
    return f"Kết quả tìm kiếm cho: {query}"


agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[search],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"write_file": True})],
)

Xem [`HumanInTheLoopMiddleware`](https://reference.langchain.com/python/langchain/agents/middleware/human_in_the_loop/HumanInTheLoopMiddleware), [Human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop).

### Các tài nguyên về middleware

- [Tổng quan về middleware](https://docs.langchain.com/oss/python/langchain/middleware/overview): Stack middleware hoạt động như thế nào và khi nào các hook được kích hoạt
- [Middleware dựng sẵn](https://docs.langchain.com/oss/python/langchain/middleware/built-in): Tài liệu tham khảo đầy đủ kèm các ví dụ cấu hình
- [Middleware tùy chỉnh](https://docs.langchain.com/oss/python/langchain/middleware/custom): Viết các hook của riêng bạn cho logic nghiệp vụ, làm sạch dữ liệu PII và hơn thế nữa